# Day 1 — Value class and manual backward

Code-along with [Karpathy's spelled-out intro](https://www.youtube.com/watch?v=VMj-3S1tku0). Stop the video as you go.

## What you ship today

1. A `Value` class that wraps a single scalar, tracks parents, and supports `+`, `*`, `tanh`.
2. A **manual** `backward()` written by hand for a specific 3-node expression. (We replace this with proper topo-sort on Day 2.)
3. A side-by-side check: same expression in PyTorch, identical gradients.

## Anti-goals

- Don't write a full topo-sort `backward()` here. That's Day 2.
- Don't add `exp`, `relu`, `**` yet.
- Don't refactor the `Value` class while typing — get the bones right first.

In [ ]:
# Sanity check — should print PyTorch version and CUDA: False on Mac (or True on Razer).
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## Step 1 — minimal `Value` with `data` and `grad`

From the lecture: the first version is a wrapper around a scalar. No autograd yet, just storage.

In [ ]:
# TODO: implement Value with .data, .grad (init 0.0), repr.
# DO NOT add backward yet. We get the shape of the class right first.
class Value:
    pass  # replace

a = Value(2.0)
b = Value(-3.0)
print(a, b)

## Step 2 — `__add__` and `__mul__`, tracking parents

When `c = a + b`, `c` needs to remember it came from `a` and `b` (we'll need that for the chain rule). Karpathy uses a `_prev` set.

In [ ]:
# TODO: extend Value with __add__, __mul__, _prev (set of parents), _op (str).
# Test: c = a + b; d = a * b; print(c._prev, d._prev, d._op)

## Step 3 — `tanh`

Why tanh and not relu yet? Tanh's derivative `1 - tanh(x)**2` only depends on the *output*, which makes the chain rule arithmetic clean for a first pass. ReLU comes Day 2.

**Whiteboard exercise (don't skip):** before typing, write the chain rule for `o = tanh(a*b + c)`. Compute `do/da`, `do/db`, `do/dc` by hand. Confirm the answers below match what you derive.

In [ ]:
import math
# TODO: add Value.tanh() method. Returns a new Value with _prev = {self}, _op='tanh'.
# Test: t = a.tanh(); print(t)

## Step 4 — manual backward pass on a fixed graph

We write `_backward` *as a closure* on each node, by hand, for one specific graph. The point is to see the chain rule mechanic before automating it.

Graph: `o = tanh(a*b + c)`.

In [ ]:
# TODO: define _backward closures on each node so that calling them in reverse
# topological order populates .grad correctly.
#
# Order: o, then the (a*b + c) node, then (a*b), then a and b and c.
# o.grad = 1.0 to seed.
# Local derivatives:
#   o = tanh(t)            -> dt = (1 - o.data**2) * o.grad
#   t = ab_node + c        -> d(ab_node) = 1 * t.grad ; dc = 1 * t.grad
#   ab_node = a * b        -> da = b.data * ab_node.grad ; db = a.data * ab_node.grad
#
# Don't write Value.backward() yet. Day 2.

## Step 5 — gradient check against PyTorch

Same expression in PyTorch. The two `.grad` numbers must match to ~1e-6.

In [ ]:
import torch
ta = torch.tensor(2.0, requires_grad=True)
tb = torch.tensor(-3.0, requires_grad=True)
tc = torch.tensor(10.0, requires_grad=True)
to_out = (ta * tb + tc).tanh()
to_out.backward()
print('PyTorch grads:', ta.grad.item(), tb.grad.item(), tc.grad.item())
# TODO: assert your micrograd grads match these to atol=1e-6

## End-of-day check

Before leaving Day 1, you should be able to answer:

1. Why does `o.grad = 1.0` seed the backward pass? (Answer: `do/do = 1`.)
2. What would happen if `_prev` was a list instead of a set? (Answer: if you used the same `Value` twice, you'd double-count the parent and double the gradient contribution. Sets dedupe.)
3. Why do we accumulate `+=` instead of assigning `.grad`? (Answer: a node can have multiple downstream consumers; each contributes via its branch of the chain rule.)

If any of these are blurry, rewatch the relevant chunk of the video before Day 2.

### Append a note to NOTES.md

One sentence: what was the most surprising thing today? (Most common answer: 'how mechanical backprop actually is.')